# SE-ResNeXt-50-32x4d: Predictive, Joint-Guided and CAM Ablation

This experiment compares KL-grading performance and faithful localization. It keeps native CAM as the production explanation, adds a weak joint-guided final-linear arm, and audits final-feature Grad-CAM as a secondary diagnostic.

The test split is never loaded. All arms use the same split, preprocessing, sampler, seed and 5/15/10 schedule.


## 0. Import lib
Import library, load device, connect to google drive

In [1]:
import os
import hashlib
from collections import Counter
from typing import List, Union
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import tqdm
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# Try mounting drive (if on Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except ImportError:
    print("Not running in Google Colab. Skipping Drive mount.")

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# Install timm if needed
try:
    import timm
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "timm"])
    import timm

# Install torchmetrics if needed
try:
    import torchmetrics
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "torchmetrics"])
    import torchmetrics

# Install seaborn if needed
try:
    import seaborn
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "seaborn"])
    import seaborn

Mounted at /content/drive
Google Drive mounted successfully.
Using device: cuda
GPU Name: Tesla T4


## 1. Prepare dataset 
unzip dataset from google drive 

In [2]:
import subprocess
import os
import torch
import numpy as np
from datetime import datetime, timezone

# Unzip dataset from Drive if running on Google Colab
dataset_zip = "/content/drive/MyDrive/Datasets/kaggle_knee_osteoarthritis.zip"
if os.path.exists(dataset_zip):
    print("Unzipping dataset from Google Drive...")
    subprocess.run(["unzip", "-q", "-o", dataset_zip, "-d", "/content/Datasets"], check=True)
else:
    print("Zip file not found at default Drive path. Assuming local dataset path.")

# =========================================================================
# VALIDATED FINAL-LINEAR-CAM CONFIGURATION
# =========================================================================
class TrainingConfig:
    # Model and explanation architecture
    model_name = "seresnext50_32x4d"
    architecture = "seresnext_cam_comparison_harness"
    pretrained = True
    num_classes = 5

    # Dataset and run-isolated checkpoints
    dataset_root = "/content/Datasets/kaggle_knee_osteoarthritis"
    checkpoint_root = "/content/drive/MyDrive/Models/seresnext50_32x4d_checkpoints"
    run_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f_UTC")
    checkpoint_dir = os.path.join(
        checkpoint_root, f"{run_timestamp}_seresnext_cam_comparison"
    )
    img_size = 400
    crop_size = 384
    batch_size = 48
    num_workers = 4
    seed = 42
    use_amp = True

    # Data policy from the winning faithful-CAM ablation
    canonicalize_laterality = True    # Mirror R knees so all joints share one orientation
    use_balanced_sampler = True
    sampler_power = 1.0               # Full inverse-frequency sampling
    use_minority_aug = False
    use_tta = False
    random_erasing_p = 0.10
    random_erasing_second_p = 0.0
    rotation_degrees = 5
    brightness_jitter = 0.08
    contrast_jitter = 0.08

    # Exact 5/15/10 training schedule used in the comparison
    training_pipeline = "3-stage"
    resume_from_last = False
    use_early_stopping = False
    early_stopping_patience_stage2 = 15
    early_stopping_patience_stage3 = 10
    stage1_epochs = 5
    stage2_epochs = 15
    stage3_epochs = 10
    total_epochs_standard = 30

    lr_warmup = 3e-4
    lr_coarse_head = 3e-4
    lr_coarse_backbone = 3e-5
    lr_finetune = 1e-5
    lr_standard = 3e-4
    weight_decay = 1e-4

    # Native grade-specific maps require one CE logit/map per KL class.
    loss_stage1 = "ce"
    loss_stage2 = "ce"
    loss_stage3 = "ce"
    loss_standard = "ce"

    # Validation-only predictive score used by the faithful-CAM ablation.
    selection_qwk_weight = 0.40
    selection_macro_f1_weight = 0.20
    selection_macro_recall_weight = 0.10
    selection_grade1_recall_weight = 0.10
    selection_macro_ap_weight = 0.15
    selection_macro_auc_weight = 0.05

    scheduler_stage2 = "cosine"
    scheduler_stage3 = "cosine"
    scheduler_standard = "cosine"


def log_config(config):
    print("=" * 65)
    print(" ACTIVE TRAINING CONFIGURATION LOG")
    print("=" * 65)
    attrs = [
        attr for attr in dir(config)
        if not attr.startswith("__") and not callable(getattr(config, attr))
    ]
    for attr in attrs:
        print(f"{attr:<32} : {getattr(config, attr)}")
    print("=" * 65)


log_config(TrainingConfig)

DATASET_ROOT_PATH = TrainingConfig.dataset_root
CHECKPOINT_SAVE_DIR = TrainingConfig.checkpoint_dir
BATCH_SIZE = TrainingConfig.batch_size
IMG_SIZE = TrainingConfig.img_size
CROP_SIZE = TrainingConfig.crop_size

torch.manual_seed(TrainingConfig.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TrainingConfig.seed)
np.random.seed(TrainingConfig.seed)
import random
random.seed(TrainingConfig.seed)

os.makedirs(CHECKPOINT_SAVE_DIR, exist_ok=False)


Unzipping dataset from Google Drive...
 ACTIVE TRAINING CONFIGURATION LOG
architecture                     : seresnext_cam_comparison_harness
batch_size                       : 48
brightness_jitter                : 0.08
canonicalize_laterality          : True
checkpoint_dir                   : /content/drive/MyDrive/Models/seresnext50_32x4d_checkpoints/2026-07-22_11-50-49_419023_UTC_seresnext_cam_comparison
checkpoint_root                  : /content/drive/MyDrive/Models/seresnext50_32x4d_checkpoints
contrast_jitter                  : 0.08
crop_size                        : 384
dataset_root                     : /content/Datasets/kaggle_knee_osteoarthritis
early_stopping_patience_stage2   : 15
early_stopping_patience_stage3   : 10
img_size                         : 400
loss_stage1                      : ce
loss_stage2                      : ce
loss_stage3                      : ce
loss_standard                    : ce
lr_coarse_backbone               : 3e-05
lr_coarse_head             

## 2. Train config loader
Load train config from input user + seed

In [3]:
# =========================================================================
# CONFIGURATION & PARAMETERS (Unified Input Config)
# =========================================================================
# TrainingConfig has been unified and defined in Section 1 (Cell 4) 
# to prevent redundant redefinition and early stopping bugs.

# Log configurations
log_config(TrainingConfig)

# Set global alias variables for compatibility with downstream cells
DATASET_ROOT_PATH = TrainingConfig.dataset_root
CHECKPOINT_SAVE_DIR = TrainingConfig.checkpoint_dir
BATCH_SIZE = TrainingConfig.batch_size
IMG_SIZE = TrainingConfig.img_size

# Set random seed for reproducibility
torch.manual_seed(TrainingConfig.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TrainingConfig.seed)
np.random.seed(TrainingConfig.seed)
import random
random.seed(TrainingConfig.seed)

os.makedirs(CHECKPOINT_SAVE_DIR, exist_ok=True)


 ACTIVE TRAINING CONFIGURATION LOG
architecture                     : seresnext_cam_comparison_harness
batch_size                       : 48
brightness_jitter                : 0.08
canonicalize_laterality          : True
checkpoint_dir                   : /content/drive/MyDrive/Models/seresnext50_32x4d_checkpoints/2026-07-22_11-50-49_419023_UTC_seresnext_cam_comparison
checkpoint_root                  : /content/drive/MyDrive/Models/seresnext50_32x4d_checkpoints
contrast_jitter                  : 0.08
crop_size                        : 384
dataset_root                     : /content/Datasets/kaggle_knee_osteoarthritis
early_stopping_patience_stage2   : 15
early_stopping_patience_stage3   : 10
img_size                         : 400
loss_stage1                      : ce
loss_stage2                      : ce
loss_stage3                      : ce
loss_standard                    : ce
lr_coarse_backbone               : 3e-05
lr_coarse_head                   : 0.0003
lr_finetune             

## 3. Preprocessing image
Padding + CLAHE + Transforms (train, val, minority) + Remove duplicate

In [4]:
class SquarePadOpenCV(object):
    """Pads a rectangular image to a square."""
    def __call__(self, image):
        h, w = image.shape[:2]
        max_wh = max(h, w)
        pad_top = (max_wh - h) // 2
        pad_bottom = max_wh - h - pad_top
        pad_left = (max_wh - w) // 2
        pad_right = max_wh - w - pad_left
        
        padded_image = cv2.copyMakeBorder(
            image, pad_top, pad_bottom, pad_left, pad_right, 
            borderType=cv2.BORDER_CONSTANT, value=[0, 0, 0]
        )
        return padded_image

class OpenCVCLAHE(object):
    """Applies CLAHE (Contrast Limited Adaptive Histogram Equalization) using OpenCV."""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img_rgb: np.ndarray) -> np.ndarray:
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        l_channel, a_channel, b_channel = cv2.split(img_lab)
        clahe_l_channel = clahe.apply(l_channel)
        merged_lab_image = cv2.merge((clahe_l_channel, a_channel, b_channel))
        return cv2.cvtColor(merged_lab_image, cv2.COLOR_LAB2RGB)

def get_transforms(img_size=400, crop_size=384):
    """Build transforms after laterality canonicalization; no random mirroring."""
    train_transform = transforms.Compose([
        SquarePadOpenCV(),
        OpenCVCLAHE(),
        transforms.ToPILImage(),
        transforms.RandomRotation(degrees=TrainingConfig.rotation_degrees),
        transforms.ColorJitter(
            brightness=TrainingConfig.brightness_jitter,
            contrast=TrainingConfig.contrast_jitter,
        ),
        transforms.Resize((img_size, img_size)),
        transforms.RandomCrop(crop_size),
        transforms.ToTensor(),
        transforms.RandomErasing(p=TrainingConfig.random_erasing_p, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
        transforms.RandomErasing(p=TrainingConfig.random_erasing_second_p, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transform = transforms.Compose([
        SquarePadOpenCV(),
        OpenCVCLAHE(),
        transforms.ToPILImage(),
        transforms.Resize((img_size, img_size)),
        transforms.CenterCrop(crop_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transform_tta = transforms.Compose([
        SquarePadOpenCV(),
        OpenCVCLAHE(),
        transforms.ToPILImage(),
        transforms.Resize((img_size, img_size)),
        transforms.FiveCrop(crop_size),
        transforms.Lambda(lambda crops: torch.stack([
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(transforms.ToTensor()(crop))
            for crop in crops
        ]))
    ])

    minority_train_transform = transforms.Compose([
        SquarePadOpenCV(),
        OpenCVCLAHE(),
        transforms.ToPILImage(),
        transforms.RandomRotation(degrees=TrainingConfig.rotation_degrees),
        transforms.ColorJitter(
            brightness=TrainingConfig.brightness_jitter,
            contrast=TrainingConfig.contrast_jitter,
        ),
        transforms.RandomAffine(degrees=0, translate=(0.04, 0.04), scale=(0.97, 1.03)),
        transforms.Resize((img_size, img_size)),
        transforms.RandomCrop(crop_size),
        transforms.ToTensor(),
        transforms.RandomErasing(p=TrainingConfig.random_erasing_p, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
        transforms.RandomErasing(p=TrainingConfig.random_erasing_second_p, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    return train_transform, val_transform, val_transform_tta, minority_train_transform
def remove_duplicate_images(image_paths: list, labels: list, exclude_hashes: set = None):
    """Removes duplicate images using MD5 hashing."""
    total_found = len(image_paths)
    unique_paths, unique_labels, unique_hashes = [], [], set()
    internal_dup_count, leakage_count = 0, 0
    
    for path, label in zip(image_paths, labels):
        hash_md5 = hashlib.md5()
        try:
            with open(path, "rb") as f:
                for chunk in iter(lambda: f.read(4096), b""): 
                    hash_md5.update(chunk)
            h = hash_md5.hexdigest()
        except Exception as e:
            print(f"Warning: Could not read image {path}: {e}")
            continue
            
        if exclude_hashes and h in exclude_hashes:
            leakage_count += 1
            continue
        if h in unique_hashes:
            internal_dup_count += 1
            continue
            
        unique_hashes.add(h)
        unique_paths.append(path)
        unique_labels.append(label)
        
    print(f"\n--- Deduplication: Files found: {total_found} | Unique kept: {len(unique_paths)} | Dupes removed: {internal_dup_count} | Cross-split leaks: {leakage_count}")
    return unique_paths, unique_labels, unique_hashes

## 4. Dataset
Load kaggle dataset (apply transform + duplicate remove)

In [5]:
def remove_duplicate_images(image_paths: list, labels: list, exclude_hashes: set = None):
    """Remove byte-identical images and cross-split duplicates using MD5."""
    total_found = len(image_paths)
    unique_paths, unique_labels, unique_hashes = [], [], set()
    internal_dup_count, leakage_count = 0, 0

    for path, label in zip(image_paths, labels):
        hash_md5 = hashlib.md5()
        try:
            with open(path, "rb") as image_file:
                for chunk in iter(lambda: image_file.read(4096), b""):
                    hash_md5.update(chunk)
            digest = hash_md5.hexdigest()
        except Exception as error:
            print(f"Warning: Could not read image {path}: {error}")
            continue

        if exclude_hashes and digest in exclude_hashes:
            leakage_count += 1
            continue
        if digest in unique_hashes:
            internal_dup_count += 1
            continue

        unique_hashes.add(digest)
        unique_paths.append(path)
        unique_labels.append(label)

    print(
        f"\n--- Deduplication: Files found: {total_found} | "
        f"Unique kept: {len(unique_paths)} | Dupes removed: {internal_dup_count} | "
        f"Cross-split leaks: {leakage_count}"
    )
    return unique_paths, unique_labels, unique_hashes


def is_right_knee_path(image_path: str) -> bool:
    """The OAI/Kaggle filename suffix identifies left (L) and right (R) knees."""
    stem = os.path.splitext(os.path.basename(image_path))[0]
    return stem.upper().endswith("R")


def canonicalize_knee_laterality(image: np.ndarray, image_path: str) -> np.ndarray:
    """Mirror right knees so medial/lateral anatomy has one consistent convention."""
    if TrainingConfig.canonicalize_laterality and is_right_knee_path(image_path):
        return np.ascontiguousarray(image[:, ::-1])
    return image


class KaggleKneeOsteoarthritisDataset(Dataset):
    """Load one split and apply identical laterality logic to every downstream path."""

    def __init__(self, root: str, split_dir: str, transform=None, exclude_hashes: set = None, minority_transform=None):
        self.root = root
        self.transform = transform
        self.minority_transform = minority_transform
        raw_paths, raw_labels = [], []
        split_path = os.path.join(root, split_dir)

        if not os.path.isdir(split_path):
            raise FileNotFoundError(f"Split directory not found: {split_path}")

        class_names = sorted(
            directory for directory in os.listdir(split_path)
            if os.path.isdir(os.path.join(split_path, directory)) and directory.isdigit()
        )
        print(f"Loading '{split_dir}' split from: {split_path}")

        for class_name in class_names:
            class_dir = os.path.join(split_path, class_name)
            label = int(class_name)
            valid_extensions = (".png", ".jpg", ".jpeg")
            image_files = [
                filename for filename in os.listdir(class_dir)
                if filename.lower().endswith(valid_extensions)
            ]
            for filename in image_files:
                raw_paths.append(os.path.join(class_dir, filename))
                raw_labels.append(label)

        self.image_paths, self.labels, self.image_hashes = remove_duplicate_images(
            raw_paths, raw_labels, exclude_hashes=exclude_hashes
        )

    def load_image_from_path(self, image_path: str) -> np.ndarray:
        image_bgr = cv2.imread(image_path)
        if image_bgr is None:
            raise IOError(f"Could not read image: {image_path}")
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        return canonicalize_knee_laterality(image, image_path)

    def __getitem__(self, index: int):
        image = self.load_image_from_path(self.image_paths[index])
        label = self.labels[index]
        if self.minority_transform and label in [3, 4]:
            image = self.minority_transform(image)
        elif self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self) -> int:
        return len(self.image_paths)


## 5. Dataloader
Prepare train, val dataloader

In [6]:
# Create transforms
train_transform, val_transform, val_transform_tta, minority_train_transform = get_transforms(
    img_size=TrainingConfig.img_size, crop_size=TrainingConfig.crop_size
)
val_loader_transform = val_transform_tta if TrainingConfig.use_tta else val_transform

# Determine minority transform based on configuration
minor_transform = minority_train_transform if TrainingConfig.use_minority_aug else None

# Load training dataset
train_dataset = KaggleKneeOsteoarthritisDataset(
    root=DATASET_ROOT_PATH, split_dir="train", transform=train_transform, minority_transform=minor_transform
)
train_hashes = set(train_dataset.image_hashes)

# Load validation dataset
val_split_dir = "val"
if not os.path.isdir(os.path.join(DATASET_ROOT_PATH, val_split_dir)):
    raise FileNotFoundError("A validation split is required; test fallback is disabled.")
val_dataset = KaggleKneeOsteoarthritisDataset(
    root=DATASET_ROOT_PATH, split_dir=val_split_dir, transform=val_loader_transform, exclude_hashes=train_hashes
)

# Imbalance handling: Calculate Class-Aware WeightedRandomSampler for training split
from torch.utils.data import WeightedRandomSampler

# Count samples of each class
class_counts = Counter(train_dataset.labels)
print(f"Training class distribution: {dict(sorted(class_counts.items()))}")

# Create training loader based on config
if TrainingConfig.use_balanced_sampler:
    print("Using WeightedRandomSampler for class balance.")
    class_weights = {
        cls: 1.0 / (count ** TrainingConfig.sampler_power)
        for cls, count in class_counts.items()
    }
    sample_weights = [class_weights[label] for label in train_dataset.labels]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=TrainingConfig.num_workers, pin_memory=True, persistent_workers=TrainingConfig.num_workers > 0)
else:
    print("Using standard shuffled DataLoader.")
    train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=TrainingConfig.num_workers, pin_memory=True, persistent_workers=TrainingConfig.num_workers > 0)

val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=TrainingConfig.num_workers, pin_memory=True, persistent_workers=TrainingConfig.num_workers > 0)

print(f"Data loaders ready. Train batches: {len(train_loader)} | Validation batches: {len(val_loader)}")

Loading 'train' split from: /content/Datasets/kaggle_knee_osteoarthritis/train

--- Deduplication: Files found: 5778 | Unique kept: 5778 | Dupes removed: 0 | Cross-split leaks: 0
Loading 'val' split from: /content/Datasets/kaggle_knee_osteoarthritis/val

--- Deduplication: Files found: 826 | Unique kept: 826 | Dupes removed: 0 | Cross-split leaks: 0
Training class distribution: {0: 2286, 1: 1046, 2: 1516, 3: 757, 4: 173}
Using WeightedRandomSampler for class balance.
Data loaders ready. Train batches: 121 | Validation batches: 18


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 1. Models and Predictive Comparison


In [7]:
# Experiment definitions, models, losses, metrics, and training loop.
import json
from datetime import datetime, timezone
from itertools import chain
from pathlib import Path

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    cohen_kappa_score,
    precision_recall_fscore_support,
    recall_score,
    roc_auc_score,
)

SERESNEXT_CONFIG = {
    "model_name": "seresnext50_32x4d",
    "arms": [
        {"name": "multiscale_mlp_hirescam_ce", "architecture": "multiscale_mlp", "loss": "ce"},
        {"name": "final_native_cam_ce", "architecture": "final_linear_cam", "loss": "ce"},
        {"name": "final_native_cam_joint_guided_005", "architecture": "final_linear_cam", "loss": "ce", "joint_guidance_weight": 0.05},
        {"name": "final_native_cam_softlabel", "architecture": "final_linear_cam", "loss": "ordinal_soft_label"},
        {"name": "fpn_native_cam_ce", "architecture": "fpn_cam", "loss": "ce"},
        {"name": "fpn_native_cam_softlabel", "architecture": "fpn_cam", "loss": "ordinal_soft_label"},
    ],
    "batch_size": 48,
    "num_workers": 4,
    "seed": 42,
    "use_amp": True,
    "soft_label_sigma": 0.70,
    "fpn_channels": 256,
    "stage_epochs": {"warmup": 5, "coarse": 15, "finetune": 10},
    "predictive_weights": {
        "qwk": 0.40,
        "macro_f1": 0.20,
        "macro_recall": 0.10,
        "grade1_recall": 0.10,
        "macro_ap": 0.15,
        "macro_auc": 0.05,
    },
    "localization_thresholds": {
        "joint_enrichment_min": 1.20,
        "border_enrichment_max": 0.85,
        "occlusion_spearman_min": 0.30,
    },
    "cam_cases_per_grade": 50,
    "test_evaluated": False,
}


def seed_experiment(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def load_checkpoint(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


class SEResNeXtTrainable(nn.Module):
    def freeze_backbone(self):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False
        for parameter in self.head_parameters():
            parameter.requires_grad = True

    def unfreeze_last_block(self):
        for parameter in self.parameters():
            parameter.requires_grad = False
        matched = 0
        for name, parameter in self.backbone.named_parameters():
            if any(token in name for token in ("layer3", "layer4", "stages.2", "stages.3")):
                parameter.requires_grad = True
                matched += parameter.numel()
        if matched == 0:
            raise RuntimeError(
                "Could not identify the final SE-ResNeXt stages. Inspect backbone.named_parameters()."
            )
        for parameter in self.head_parameters():
            parameter.requires_grad = True

    def unfreeze_backbone(self):
        for parameter in self.parameters():
            parameter.requires_grad = True


class MultiScaleMLPHiResCAM(SEResNeXtTrainable):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            SERESNEXT_CONFIG["model_name"],
            pretrained=pretrained,
            features_only=True,
            out_indices=(2, 3, 4),
        )
        channels = list(self.backbone.feature_info.channels())
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(sum(channels), 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(512, 5),
        )

    def head_parameters(self):
        return self.classifier.parameters()

    def forward_features(self, images):
        return self.backbone(images)

    def logits_from_features(self, features):
        pooled = [self.gap(feature).flatten(1) for feature in features]
        return self.classifier(torch.cat(pooled, dim=1))

    def forward(self, images):
        return self.logits_from_features(self.forward_features(images))

    def explain(self, images, class_index):
        with torch.enable_grad():
            features = self.forward_features(images)
            logits = self.logits_from_features(features)
            gradients = torch.autograd.grad(logits[0, class_index], features)
            target_size = max(
                (feature.shape[-2:] for feature in features),
                key=lambda size: size[0] * size[1],
            )
            contributions = []
            for feature, gradient in zip(features, gradients):
                contribution = (feature * gradient).sum(dim=1, keepdim=True)
                contributions.append(
                    F.interpolate(
                        contribution, target_size, mode="bilinear", align_corners=False
                    )
                )
            cam = F.relu(torch.stack(contributions).sum(dim=0))[0, 0]
            cam = F.interpolate(
                cam[None, None], images.shape[-2:], mode="bilinear", align_corners=False
            )[0, 0]
        return cam.detach().cpu().numpy() / (cam.max().item() + 1e-8)


class FinalLinearCAM(SEResNeXtTrainable):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            SERESNEXT_CONFIG["model_name"],
            pretrained=pretrained,
            features_only=True,
            out_indices=(4,),
        )
        channels = self.backbone.feature_info.channels()[0]
        self.class_conv = nn.Conv2d(channels, 5, kernel_size=1)

    def head_parameters(self):
        return self.class_conv.parameters()

    def class_maps(self, images):
        return self.class_conv(self.backbone(images)[0])

    def forward(self, images):
        return self.class_maps(images).mean(dim=(2, 3))

    def explain(self, images, class_index):
        with torch.no_grad():
            cam = F.relu(self.class_maps(images)[0, class_index])
            cam = F.interpolate(
                cam[None, None], images.shape[-2:], mode="bilinear", align_corners=False
            )[0, 0]
        return cam.cpu().numpy() / (cam.max().item() + 1e-8)


class FPNLinearCAM(SEResNeXtTrainable):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            SERESNEXT_CONFIG["model_name"],
            pretrained=pretrained,
            features_only=True,
            out_indices=(2, 3, 4),
        )
        channels = list(self.backbone.feature_info.channels())
        width = SERESNEXT_CONFIG["fpn_channels"]
        self.projections = nn.ModuleList(
            [nn.Conv2d(channel, width, kernel_size=1) for channel in channels]
        )
        self.refine = nn.Sequential(
            nn.Conv2d(width, width, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(width),
            nn.ReLU(),
        )
        self.class_conv = nn.Conv2d(width, 5, kernel_size=1)

    def head_parameters(self):
        return chain(
            self.projections.parameters(),
            self.refine.parameters(),
            self.class_conv.parameters(),
        )

    def class_maps(self, images):
        features = self.backbone(images)
        target_size = features[0].shape[-2:]
        projected = []
        for projection, feature in zip(self.projections, features):
            value = projection(feature)
            if value.shape[-2:] != target_size:
                value = F.interpolate(
                    value, target_size, mode="bilinear", align_corners=False
                )
            projected.append(value)
        return self.class_conv(self.refine(torch.stack(projected).sum(dim=0)))

    def forward(self, images):
        return self.class_maps(images).mean(dim=(2, 3))

    def explain(self, images, class_index):
        with torch.no_grad():
            cam = F.relu(self.class_maps(images)[0, class_index])
            cam = F.interpolate(
                cam[None, None], images.shape[-2:], mode="bilinear", align_corners=False
            )[0, 0]
        return cam.cpu().numpy() / (cam.max().item() + 1e-8)


def build_model(architecture, pretrained):
    if architecture == "multiscale_mlp":
        return MultiScaleMLPHiResCAM(pretrained)
    if architecture == "final_linear_cam":
        return FinalLinearCAM(pretrained)
    if architecture == "fpn_cam":
        return FPNLinearCAM(pretrained)
    raise ValueError(f"Unknown architecture: {architecture}")


def ordinal_soft_label_loss(logits, labels):
    grades = torch.arange(5, dtype=logits.dtype, device=logits.device)
    distances = grades.unsqueeze(0) - labels.to(logits.dtype).unsqueeze(1)
    targets = torch.exp(
        -0.5 * (distances / SERESNEXT_CONFIG["soft_label_sigma"]) ** 2
    )
    targets = targets / targets.sum(dim=1, keepdim=True)
    return -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()


def calculate_loss(loss_name, logits, labels):
    if loss_name == "ce":
        return F.cross_entropy(logits, labels)
    if loss_name == "ordinal_soft_label":
        return ordinal_soft_label_loss(logits, labels)
    raise ValueError(loss_name)


def calculate_metrics(labels, predictions, probabilities):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    probabilities = np.asarray(probabilities)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    one_hot = np.eye(5)[labels]
    per_class_ap = average_precision_score(one_hot, probabilities, average=None)
    per_class_auc = roc_auc_score(one_hot, probabilities, average=None)
    metrics = {
        "accuracy": float(accuracy_score(labels, predictions)),
        "qwk": float(cohen_kappa_score(labels, predictions, weights="quadratic")),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
        "grade1_recall": float(
            recall_score(labels == 1, predictions == 1, zero_division=0)
        ),
        "macro_ap": float(per_class_ap.mean()),
        "macro_auc": float(per_class_auc.mean()),
    }
    for grade in range(5):
        metrics[f"grade{grade}_ap"] = float(per_class_ap[grade])
        metrics[f"grade{grade}_auc"] = float(per_class_auc[grade])
    metrics["predictive_score"] = float(
        sum(
            SERESNEXT_CONFIG["predictive_weights"][key] * metrics[key]
            for key in SERESNEXT_CONFIG["predictive_weights"]
        )
    )
    return metrics


def evaluate_model(model, loader, loss_name, return_arrays=False):
    model.eval()
    total_loss = 0.0
    labels_all, predictions_all, probabilities_all = [], [], []
    with torch.no_grad():
        for images, labels in tqdm.tqdm(loader, desc="VALIDATE", leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.amp.autocast(
                device_type=device.type,
                enabled=SERESNEXT_CONFIG["use_amp"] and device.type == "cuda",
            ):
                logits = model(images)
                loss = calculate_loss(loss_name, logits, labels)
            probabilities = F.softmax(logits.float(), dim=1)
            total_loss += loss.item() * labels.size(0)
            labels_all.extend(labels.cpu().numpy())
            predictions_all.extend(probabilities.argmax(dim=1).cpu().numpy())
            probabilities_all.extend(probabilities.cpu().numpy())
    metrics = calculate_metrics(labels_all, predictions_all, probabilities_all)
    metrics["val_loss"] = total_loss / len(loader.dataset)
    if return_arrays:
        return (
            metrics,
            np.asarray(labels_all),
            np.asarray(predictions_all),
            np.asarray(probabilities_all),
        )
    return metrics



def joint_guidance_loss(class_maps, labels):
    """Weakly prefer true-grade evidence near the tibiofemoral band."""
    batch_size, _, height, width = class_maps.shape
    selected = class_maps[
        torch.arange(batch_size, device=class_maps.device), labels
    ]
    positive = F.softplus(selected.float())
    y = torch.linspace(0, 1, height, device=class_maps.device).view(1, height, 1)
    x = torch.linspace(0, 1, width, device=class_maps.device).view(1, 1, width)
    soft_joint = torch.exp(-0.5 * ((y - 0.50) / 0.16) ** 2).expand(1, height, width)
    border = ((x < 0.08) | (x > 0.92) | (y < 0.08) | (y > 0.92)).float()
    cost = (1.0 - soft_joint) + 0.20 * border
    return (
        (positive * cost).flatten(1).sum(1)
        / positive.flatten(1).sum(1).clamp_min(1e-8)
    ).mean()

def train_epoch(model, loader, optimizer, scaler, loss_name, description, guidance_weight=0.0):
    model.train()
    total_loss = 0.0
    for images, labels in tqdm.tqdm(loader, desc=description, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(
            device_type=device.type,
            enabled=SERESNEXT_CONFIG["use_amp"] and device.type == "cuda",
        ):
            if guidance_weight > 0.0 and hasattr(model, "class_maps"):
                class_maps = model.class_maps(images)
                logits = class_maps.mean(dim=(2, 3))
                guidance = joint_guidance_loss(class_maps, labels)
            else:
                logits = model(images)
                guidance = torch.zeros((), device=images.device)
            loss = calculate_loss(loss_name, logits, labels) + guidance_weight * guidance
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * labels.size(0)
    return total_loss / len(loader.dataset)


def configure_stage(model, stage, epochs):
    if stage == "warmup":
        model.freeze_backbone()
        optimizer = optim.AdamW(
            list(model.head_parameters()),
            lr=TrainingConfig.lr_warmup,
            weight_decay=TrainingConfig.weight_decay,
        )
        return optimizer, None
    if stage == "coarse":
        model.unfreeze_last_block()
        optimizer = optim.AdamW(
            [
                {
                    "params": list(
                        filter(lambda parameter: parameter.requires_grad, model.backbone.parameters())
                    ),
                    "lr": TrainingConfig.lr_coarse_backbone,
                },
                {
                    "params": list(model.head_parameters()),
                    "lr": TrainingConfig.lr_coarse_head,
                },
            ],
            weight_decay=TrainingConfig.weight_decay,
        )
    else:
        model.unfreeze_backbone()
        optimizer = optim.AdamW(
            model.parameters(),
            lr=TrainingConfig.lr_finetune,
            weight_decay=10 * TrainingConfig.weight_decay,
        )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-7
    )
    return optimizer, scheduler


train_data = train_dataset
val_data = val_dataset
counts = np.bincount(np.asarray(train_data.labels), minlength=5).astype(float)
sample_weights = torch.as_tensor(
    [1.0 / counts[label] for label in train_data.labels], dtype=torch.double
)


def run_arm(arm, batch_dir):
    seed_experiment(SERESNEXT_CONFIG["seed"])
    run_dir = batch_dir / arm["name"]
    run_dir.mkdir(parents=True, exist_ok=False)
    generator = torch.Generator().manual_seed(SERESNEXT_CONFIG["seed"])
    sampler = WeightedRandomSampler(
        sample_weights,
        len(sample_weights),
        replacement=True,
        generator=generator,
    )
    train_loader_arm = DataLoader(
        train_data,
        batch_size=SERESNEXT_CONFIG["batch_size"],
        sampler=sampler,
        num_workers=SERESNEXT_CONFIG["num_workers"],
        pin_memory=True,
        persistent_workers=SERESNEXT_CONFIG["num_workers"] > 0,
        worker_init_fn=seed_worker,
        generator=generator,
    )
    val_loader_arm = DataLoader(
        val_data,
        batch_size=SERESNEXT_CONFIG["batch_size"],
        shuffle=False,
        num_workers=SERESNEXT_CONFIG["num_workers"],
        pin_memory=True,
        persistent_workers=SERESNEXT_CONFIG["num_workers"] > 0,
        worker_init_fn=seed_worker,
    )
    model = build_model(arm["architecture"], pretrained=True).to(device)
    scaler = torch.amp.GradScaler(
        "cuda", enabled=SERESNEXT_CONFIG["use_amp"] and device.type == "cuda"
    )
    stage2_path = run_dir / "stage2_best_model.pth"
    best_path = run_dir / "best_model.pth"
    best_stage2 = best_final = -np.inf
    best_payload = None
    history = []
    epoch = 0

    for stage, epochs in SERESNEXT_CONFIG["stage_epochs"].items():
        if stage == "finetune" and stage2_path.exists():
            stage2_checkpoint = load_checkpoint(stage2_path)
            model.load_state_dict(stage2_checkpoint["model_state_dict"])
        optimizer, scheduler = configure_stage(model, stage, epochs)
        for stage_epoch in range(epochs):
            epoch += 1
            train_loss = train_epoch(
                model,
                train_loader_arm,
                optimizer,
                scaler,
                arm["loss"],
                f"{arm['name']} {stage} {stage_epoch + 1}/{epochs}",
                guidance_weight=float(arm.get("joint_guidance_weight", 0.0)),
            )
            metrics = evaluate_model(model, val_loader_arm, arm["loss"])
            history.append(
                {"epoch": epoch, "stage": stage, "train_loss": train_loss, **metrics}
            )
            print(
                f"{arm['name']} epoch={epoch}: QWK={metrics['qwk']:.4f}, "
                f"F1={metrics['macro_f1']:.4f}, AP={metrics['macro_ap']:.4f}, "
                f"score={metrics['predictive_score']:.4f}"
            )
            if scheduler is not None:
                scheduler.step()
            payload = {
                "model_state_dict": model.state_dict(),
                "arm": arm,
                "epoch": epoch,
                "metrics": metrics,
                "config": SERESNEXT_CONFIG,
            }
            if stage == "coarse" and metrics["predictive_score"] > best_stage2:
                best_stage2 = metrics["predictive_score"]
                torch.save(payload, stage2_path)
            if stage == "finetune" and metrics["predictive_score"] > best_final:
                best_final = metrics["predictive_score"]
                best_payload = payload
                torch.save(payload, best_path)

    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    if best_payload is None:
        raise RuntimeError(f"No final checkpoint was selected for {arm['name']}")
    best_checkpoint = load_checkpoint(best_path)
    model.load_state_dict(best_checkpoint["model_state_dict"])
    metrics, labels, predictions, probabilities = evaluate_model(
        model, val_loader_arm, arm["loss"], return_arrays=True
    )
    np.savez_compressed(
        run_dir / "validation_predictions.npz",
        labels=labels,
        predictions=predictions,
        probabilities=probabilities,
    )
    result = {
        "arm": arm["name"],
        "architecture": arm["architecture"],
        "loss": arm["loss"],
        "joint_guidance_weight": float(arm.get("joint_guidance_weight", 0.0)),
        "best_epoch": best_checkpoint["epoch"],
        **metrics,
        "checkpoint": str(best_path),
        "run_dir": str(run_dir),
    }
    (run_dir / "best_validation_metrics.json").write_text(
        json.dumps(result, indent=2), encoding="utf-8"
    )
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


created_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f_UTC")
SERESNEXT_BATCH_DIR = (
    Path(TrainingConfig.checkpoint_root) / "seresnext50_cam_ablations" / stamp
)
SERESNEXT_BATCH_DIR.mkdir(parents=True, exist_ok=False)
(SERESNEXT_BATCH_DIR / "experiment_config.json").write_text(
    json.dumps(SERESNEXT_CONFIG, indent=2), encoding="utf-8"
)
seresnext_results = [
    run_arm(arm, SERESNEXT_BATCH_DIR) for arm in SERESNEXT_CONFIG["arms"]
]
seresnext_results_df = (
    pd.DataFrame(seresnext_results)
    .sort_values("predictive_score", ascending=False)
    .reset_index(drop=True)
)
seresnext_results_df.to_csv(
    SERESNEXT_BATCH_DIR / "predictive_comparison.csv", index=False
)
display(
    seresnext_results_df[
        [
            "arm", "accuracy", "qwk", "macro_precision", "macro_recall",
            "macro_f1", "grade1_recall", "macro_ap", "macro_auc",
            "predictive_score",
        ]
    ]
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  111MB            

model.safetensors: downloading bytes:           |  0.00B            

multiscale_mlp_hirescam_ce warmup 1/5:   0%|          | 0/121 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


multiscale_mlp_hirescam_ce epoch=1: QWK=0.0000, F1=0.0455, AP=0.3381, score=0.1130


...

fpn_native_cam_softlabel epoch=27: QWK=0.7824, F1=0.6244, AP=0.6783, score=0.6899


fpn_native_cam_softlabel epoch=28: QWK=0.7766, F1=0.6115, AP=0.6780, score=0.6840


fpn_native_cam_softlabel epoch=29: QWK=0.7745, F1=0.6189, AP=0.6791, score=0.6861


fpn_native_cam_softlabel epoch=30: QWK=0.7703, F1=0.6102, AP=0.6798, score=0.6799


,arm,accuracy,qwk,macro_precision,macro_recall,macro_f1,grade1_recall,macro_ap,macro_auc,predictive_score
0,final_native_cam_ce,0.618644,0.784774,0.649550,0.670463,0.656100,0.444444,0.708521,0.879008,0.706849
1,final_native_cam_softlabel,0.604116,0.797587,0.647898,0.662470,0.648468,0.437908,0.697257,0.869670,0.706838
2,final_native_cam_joint_guided_005,0.623487,0.777967,0.644049,0.670564,0.654498,0.431373,0.701474,0.877178,0.701360
3,multiscale_mlp_hirescam_ce,0.612591,0.778239,0.628675,0.664842,0.643774,0.366013,0.700195,0.872315,0.691781
4,fpn_native_cam_softlabel,0.589588,0.782447,0.633357,0.626816,0.624408,0.444444,0.678323,0.863415,0.689906
5,fpn_native_cam_ce,0.573850,0.747960,0.600420,0.609358,0.602090,0.366013,0.667275,0.862846,0.660372


## 2. Heatmap Localization and Faithfulness


In [8]:
# Same-case heatmap localization and occlusion-faithfulness audit.
def load_model_for_result(row):
    model = build_model(row["architecture"], pretrained=False).to(device)
    checkpoint = load_checkpoint(row["checkpoint"])
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model


def anatomical_proxy_masks(height, width):
    # Central tibiofemoral band proxy; this is not an expert segmentation mask.
    joint = np.zeros((height, width), dtype=bool)
    joint[int(0.28 * height):int(0.72 * height), int(0.06 * width):int(0.94 * width)] = True
    border = np.ones((height, width), dtype=bool)
    border[int(0.08 * height):int(0.92 * height), int(0.08 * width):int(0.92 * width)] = False
    return joint, border


def localization_metrics(model, tensor, target, cam):
    joint, border = anatomical_proxy_masks(*cam.shape)
    total = cam.sum() + 1e-8
    joint_energy = float(cam[joint].sum() / total)
    border_energy = float(cam[border].sum() / total)
    joint_mask = torch.as_tensor(joint, device=tensor.device)[None, None]
    border_mask = torch.as_tensor(border, device=tensor.device)[None, None]

    grid_batch, cells = [], []
    grid_size = 4
    height, width = cam.shape
    for row in range(grid_size):
        for column in range(grid_size):
            y0, y1 = row * height // grid_size, (row + 1) * height // grid_size
            x0, x1 = column * width // grid_size, (column + 1) * width // grid_size
            occluded = tensor.clone()
            occluded[:, :, y0:y1, x0:x1] = 0.0
            grid_batch.append(occluded)
            cells.append((y0, y1, x0, x1))

    with torch.no_grad():
        base = F.softmax(model(tensor), dim=1)[0, target]
        joint_drop = base - F.softmax(
            model(tensor.masked_fill(joint_mask, 0.0)), dim=1
        )[0, target]
        border_drop = base - F.softmax(
            model(tensor.masked_fill(border_mask, 0.0)), dim=1
        )[0, target]
        grid_probability = F.softmax(model(torch.cat(grid_batch)), dim=1)[:, target]
    drops = (base - grid_probability).cpu().numpy()
    cam_cells = np.asarray(
        [cam[y0:y1, x0:x1].mean() for y0, y1, x0, x1 in cells]
    )
    drop_ranks = np.argsort(np.argsort(drops)).astype(float)
    cam_ranks = np.argsort(np.argsort(cam_cells)).astype(float)
    if drop_ranks.std() == 0 or cam_ranks.std() == 0:
        correlation = 0.0
    else:
        correlation = float(np.corrcoef(drop_ranks, cam_ranks)[0, 1])
    return {
        "joint_enrichment": joint_energy / float(joint.mean()),
        "border_enrichment": border_energy / float(border.mean()),
        "joint_occlusion_drop": float(joint_drop.item()),
        "border_occlusion_drop": float(border_drop.item()),
        "occlusion_spearman": correlation,
    }


def display_tensor(tensor):
    image = tensor[0].detach().cpu()
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    return (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


rng = np.random.default_rng(SERESNEXT_CONFIG["seed"])
audit_indices = []
labels_array = np.asarray(val_data.labels)
audited_per_grade = {}
for grade in range(5):
    indices = np.flatnonzero(labels_array == grade)
    rng.shuffle(indices)
    chosen = indices[:min(SERESNEXT_CONFIG["cam_cases_per_grade"], len(indices))]
    audit_indices.extend(chosen.tolist())
    audited_per_grade[grade] = len(chosen)
print(f"Validation CAM cases per grade: {audited_per_grade}")
if any(count < SERESNEXT_CONFIG["cam_cases_per_grade"] for count in audited_per_grade.values()):
    print("Note: every available validation case is used when a grade has fewer than 50 cases.")

localization_summaries = []
all_cases = []
for _, result in seresnext_results_df.iterrows():
    model = load_model_for_result(result)
    cases = []
    for index in tqdm.tqdm(audit_indices, desc=f"CAM audit: {result['arm']}"):
        image, true_grade = val_data[index]
        tensor = image[None].to(device)
        with torch.no_grad():
            predicted_grade = int(model(tensor).argmax(dim=1).item())
        cam = model.explain(tensor, predicted_grade)
        row = {
            "arm": result["arm"],
            "dataset_index": index,
            "true_grade": int(true_grade),
            "predicted_grade": predicted_grade,
            **localization_metrics(model, tensor, predicted_grade, cam),
        }
        cases.append(row)
        all_cases.append(row)

    frame = pd.DataFrame(cases)
    localization_summaries.append(
        {
            "arm": result["arm"],
            "audited_cases": len(frame),
            "joint_enrichment": frame["joint_enrichment"].mean(),
            "border_enrichment": frame["border_enrichment"].mean(),
            "joint_occlusion_drop": frame["joint_occlusion_drop"].mean(),
            "border_occlusion_drop": frame["border_occlusion_drop"].mean(),
            "occlusion_spearman": frame["occlusion_spearman"].mean(),
        }
    )

    review_dir = Path(result["run_dir"]) / "cam_review"
    review_dir.mkdir(exist_ok=True)
    review = pd.concat(
        [
            frame.sort_values("border_enrichment", ascending=False).head(3),
            frame[frame["true_grade"] != frame["predicted_grade"]].head(5),
        ]
    ).drop_duplicates("dataset_index").head(8)
    for _, case in review.iterrows():
        image, true_grade = val_data[int(case["dataset_index"])]
        tensor = image[None].to(device)
        predicted = int(case["predicted_grade"])
        predicted_cam = model.explain(tensor, predicted)
        true_cam = model.explain(tensor, int(true_grade))
        figure, axes = plt.subplots(1, 3, figsize=(15, 5))
        display_image = display_tensor(tensor)
        axes[0].imshow(display_image)
        axes[0].set_title(f"True G{true_grade}, predicted G{predicted}")
        axes[1].imshow(display_image)
        axes[1].imshow(predicted_cam, cmap="jet", alpha=0.4)
        axes[1].set_title("Predicted-class map")
        axes[2].imshow(display_image)
        axes[2].imshow(true_cam, cmap="jet", alpha=0.4)
        axes[2].set_title("True-class map")
        for axis in axes:
            axis.axis("off")
        figure.tight_layout()
        figure.savefig(
            review_dir
            / f"idx-{int(case['dataset_index'])}_true-G{true_grade}_pred-G{predicted}.png",
            dpi=160,
            bbox_inches="tight",
        )
        plt.close(figure)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pd.DataFrame(all_cases).to_csv(
    SERESNEXT_BATCH_DIR / "localization_cases.csv", index=False
)
comparison = seresnext_results_df.merge(
    pd.DataFrame(localization_summaries), on="arm"
)
thresholds = SERESNEXT_CONFIG["localization_thresholds"]
comparison["localization_pass"] = (
    (comparison["joint_enrichment"] >= thresholds["joint_enrichment_min"])
    & (comparison["border_enrichment"] <= thresholds["border_enrichment_max"])
    & (comparison["joint_occlusion_drop"] > comparison["border_occlusion_drop"])
    & (comparison["occlusion_spearman"] >= thresholds["occlusion_spearman_min"])
)
eligible = comparison[comparison["localization_pass"]]
if eligible.empty:
    SELECTED_SERESNEXT = None
    print("No arm passed every localization gate; do not promote a model yet.")
else:
    SELECTED_SERESNEXT = (
        eligible.sort_values("predictive_score", ascending=False).iloc[0].to_dict()
    )
comparison.to_csv(
    SERESNEXT_BATCH_DIR / "metric_localization_comparison.csv", index=False
)
display(
    comparison[
        [
            "arm", "qwk", "macro_f1", "macro_ap", "macro_auc",
            "joint_enrichment", "border_enrichment", "occlusion_spearman",
            "joint_occlusion_drop", "border_occlusion_drop", "localization_pass",
        ]
    ]
)
print(
    f"Selected SE-ResNeXt model: "
    f"{SELECTED_SERESNEXT['arm'] if SELECTED_SERESNEXT else 'NONE'}"
)


Validation CAM cases per grade: {0: 50, 1: 50, 2: 50, 3: 50, 4: 27}
Note: every available validation case is used when a grade has fewer than 50 cases.


CAM audit: fpn_native_cam_ce: 100%|██████████| 227/227 [01:38<00:00,  2.31it/s]


,arm,qwk,macro_f1,macro_ap,macro_auc,joint_enrichment,border_enrichment,occlusion_spearman,joint_occlusion_drop,border_occlusion_drop,localization_pass
0,final_native_cam_ce,0.784774,0.656100,0.708521,0.879008,2.243207,0.238941,0.445815,0.571768,0.254736,True
1,final_native_cam_softlabel,0.797587,0.648468,0.697257,0.869670,2.124767,0.316011,0.508914,0.330587,0.158792,True
2,final_native_cam_joint_guided_005,0.777967,0.654498,0.701474,0.877178,2.335776,0.212358,0.419539,0.567557,0.249813,True
3,multiscale_mlp_hirescam_ce,0.778239,0.643774,0.700195,0.872315,2.006465,0.441706,0.405856,0.516406,0.233458,True
4,fpn_native_cam_softlabel,0.782447,0.624408,0.678323,0.863415,1.942324,0.500219,0.449598,0.247794,0.127755,True
5,fpn_native_cam_ce,0.747960,0.602090,0.667275,0.862846,1.774219,0.530910,0.391138,0.451847,0.323919,True


Selected SE-ResNeXt model: final_native_cam_ce


## 3. Selection Report and Paired Bootstrap


In [9]:
# Export a concise report and paired validation bootstrap comparisons.
bootstrap_rows = []
if SELECTED_SERESNEXT is not None:
    selected_arrays = np.load(
        Path(SELECTED_SERESNEXT["run_dir"]) / "validation_predictions.npz"
    )
    selected_labels = selected_arrays["labels"]
    selected_predictions = selected_arrays["predictions"]
    rng = np.random.default_rng(SERESNEXT_CONFIG["seed"])
    bootstrap_indices = [
        rng.choice(len(selected_labels), len(selected_labels), replace=True)
        for _ in range(1000)
    ]
    for _, candidate in comparison.iterrows():
        candidate_arrays = np.load(
            Path(candidate["run_dir"]) / "validation_predictions.npz"
        )
        if not np.array_equal(selected_labels, candidate_arrays["labels"]):
            raise RuntimeError("Validation order differs between arms; paired bootstrap is invalid.")
        qwk_differences, f1_differences = [], []
        for indices in bootstrap_indices:
            labels = selected_labels[indices]
            if len(np.unique(labels)) < 2:
                continue
            selected_pred = selected_predictions[indices]
            candidate_pred = candidate_arrays["predictions"][indices]
            qwk_differences.append(
                cohen_kappa_score(labels, selected_pred, weights="quadratic")
                - cohen_kappa_score(labels, candidate_pred, weights="quadratic")
            )
            selected_f1 = precision_recall_fscore_support(
                labels, selected_pred, average="macro", zero_division=0
            )[2]
            candidate_f1 = precision_recall_fscore_support(
                labels, candidate_pred, average="macro", zero_division=0
            )[2]
            f1_differences.append(selected_f1 - candidate_f1)
        bootstrap_rows.append(
            {
                "selected_arm": SELECTED_SERESNEXT["arm"],
                "candidate_arm": candidate["arm"],
                "qwk_difference": float(np.mean(qwk_differences)),
                "qwk_difference_ci_low": float(np.percentile(qwk_differences, 2.5)),
                "qwk_difference_ci_high": float(np.percentile(qwk_differences, 97.5)),
                "macro_f1_difference": float(np.mean(f1_differences)),
                "macro_f1_difference_ci_low": float(np.percentile(f1_differences, 2.5)),
                "macro_f1_difference_ci_high": float(np.percentile(f1_differences, 97.5)),
            }
        )

bootstrap_frame = pd.DataFrame(bootstrap_rows)
bootstrap_frame.to_csv(
    SERESNEXT_BATCH_DIR / "paired_validation_bootstrap.csv", index=False
)

columns = [
    "arm", "qwk", "macro_f1", "grade1_recall", "macro_ap", "macro_auc",
    "joint_enrichment", "border_enrichment", "occlusion_spearman",
    "joint_occlusion_drop", "border_occlusion_drop", "localization_pass",
]
table = [
    "| " + " | ".join(columns) + " |",
    "| " + " | ".join(["---"] * len(columns)) + " |",
]
for values in comparison[columns].itertuples(index=False, name=None):
    table.append(
        "| "
        + " | ".join(
            f"{value:.4f}" if isinstance(value, (float, np.floating)) else str(value)
            for value in values
        )
        + " |"
    )

report = [
    "# SE-ResNeXt-50 Metric and Faithful-CAM Ablation",
    "",
    f"Created: `{created_at}`",
    "",
    "The test split was not loaded or evaluated.",
    "",
    "Laterality was canonicalized before augmentation. All arms used the same validation cases, full inverse-frequency sampler, 5/15/10 schedule, and predictive score.",
    "",
    "The joint mask is a central-band proxy rather than an expert anatomical segmentation. Grade 4 has fewer than 50 validation images, so all available Grade 4 validation cases were audited.",
    "",
    *table,
    "",
    f"Selected: `{SELECTED_SERESNEXT['arm'] if SELECTED_SERESNEXT else 'NONE'}`",
    "",
    "Promotion rule: pass every localization gate first, then maximize the validation predictive score. If no arm passes, no model is selected.",
]
(SERESNEXT_BATCH_DIR / "seresnext_cam_ablation_report.md").write_text(
    "\n".join(report) + "\n", encoding="utf-8"
)
print(f"Report: {SERESNEXT_BATCH_DIR / 'seresnext_cam_ablation_report.md'}")
display(bootstrap_frame)


Report: /content/drive/MyDrive/Models/seresnext50_32x4d_checkpoints/seresnext50_cam_ablations/2026-07-22_11-50-51_226627_UTC/seresnext_cam_ablation_report.md


,selected_arm,candidate_arm,qwk_difference,qwk_difference_ci_low,qwk_difference_ci_high,macro_f1_difference,macro_f1_difference_ci_low,macro_f1_difference_ci_high
0,final_native_cam_ce,final_native_cam_ce,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,final_native_cam_ce,final_native_cam_softlabel,-0.013138,-0.030788,0.002210,0.007224,-0.015972,0.031654
2,final_native_cam_ce,final_native_cam_joint_guided_005,0.006796,-0.001233,0.014999,0.001544,-0.010437,0.015753
3,final_native_cam_ce,multiscale_mlp_hirescam_ce,0.006146,-0.008530,0.021462,0.011807,-0.010407,0.034296
4,final_native_cam_ce,fpn_native_cam_softlabel,0.001992,-0.019077,0.022160,0.032445,-0.001003,0.067530
5,final_native_cam_ce,fpn_native_cam_ce,0.036897,0.017994,0.057140,0.054523,0.021817,0.088692


## Native CAM versus gradient Grad-CAM audit

The production explanation remains the native class map because its spatial mean is exactly the predicted CE logit. This section adds gradient Grad-CAM only as a secondary sanity check on the selected final-linear arm. It does not replace the native map or change the prediction API.

The comparison uses the same validation cases and final semantic feature map. It reports joint-band energy, border energy, peak-in-joint rate, and map agreement. A gradient map that looks more plausible is not automatically more faithful; occlusion and logit sensitivity remain the deciding tests.

In [11]:
# Secondary explanation audit: native class map versus gradient Grad-CAM.
class FinalFeatureGradCAM:
    def __init__(self, model):
        self.model = model
        self.features = None
        self.handle = model.backbone.register_forward_hook(self._capture)

    def _capture(self, module, inputs, output):
        self.features = output[0]
        # Inference forwards run under no_grad; only Grad-CAM needs retention.
        if self.features.requires_grad:
            self.features.retain_grad()

    def remove(self):
        self.handle.remove()

    def __call__(self, tensor, class_index):
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = self.model(tensor)
            logits[0, class_index].backward()
        features = self.features
        weights = features.grad.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * features).sum(dim=1, keepdim=True))
        cam = F.interpolate(
            cam, size=tensor.shape[-2:], mode="bilinear", align_corners=False
        )[0, 0]
        cam = cam.detach().cpu()
        return cam.numpy() / (float(cam.max()) + 1e-8)


def simple_cam_geometry(cam):
    joint, border = anatomical_proxy_masks(*cam.shape)
    total = float(cam.sum()) + 1e-8
    peak = np.unravel_index(np.argmax(cam), cam.shape)
    return {
        "joint_enrichment": float(cam[joint].sum() / total) / float(joint.mean()),
        "border_enrichment": float(cam[border].sum() / total) / float(border.mean()),
        "peak_inside_joint": float(joint[peak]),
    }


if SELECTED_SERESNEXT is not None:
    selected_model = load_model_for_result(SELECTED_SERESNEXT)
    gradcam = FinalFeatureGradCAM(selected_model)
    method_rows = []
    gallery_cases = []
    for index in tqdm.tqdm(audit_indices, desc="Native versus Grad-CAM"):
        image, true_grade = val_data[index]
        tensor = image[None].to(device)
        with torch.no_grad():
            predicted_grade = int(selected_model(tensor).argmax(dim=1).item())
            native = selected_model.explain(tensor, predicted_grade)
        gradient = gradcam(tensor, predicted_grade)
        native_geometry = simple_cam_geometry(native)
        gradient_geometry = simple_cam_geometry(gradient)
        if native.std() < 1e-8 or gradient.std() < 1e-8:
            agreement = 0.0
        else:
            agreement = float(np.corrcoef(native.ravel(), gradient.ravel())[0, 1])
        method_rows.extend([
            {
                "method": "native_cam",
                "dataset_index": index,
                "true_grade": int(true_grade),
                "predicted_grade": predicted_grade,
                "map_agreement_with_native": 1.0,
                **native_geometry,
            },
            {
                "method": "gradient_gradcam",
                "dataset_index": index,
                "true_grade": int(true_grade),
                "predicted_grade": predicted_grade,
                "map_agreement_with_native": agreement,
                **gradient_geometry,
            },
        ])
        if len(gallery_cases) < 4:
            gallery_cases.append(
                (index, image, int(true_grade), predicted_grade, native, gradient)
            )
    gradcam.remove()
    method_frame = pd.DataFrame(method_rows)
    method_frame.to_csv(
        SERESNEXT_BATCH_DIR / "native_vs_gradient_gradcam.csv", index=False
    )
    method_summary = method_frame.groupby("method")[[
        "joint_enrichment", "border_enrichment", "peak_inside_joint"
    ]].mean()
    method_summary.to_csv(
        SERESNEXT_BATCH_DIR / "native_vs_gradient_gradcam_summary.csv"
    )
    print(method_summary)

    figure, axes = plt.subplots(
        len(gallery_cases), 3, figsize=(15, 5 * len(gallery_cases))
    )
    if len(gallery_cases) == 1:
        axes = np.asarray([axes])
    for row_index, (index, image, true_grade, predicted_grade, native, gradient) in enumerate(gallery_cases):
        display_image = display_tensor(image[None].to(device))
        axes[row_index, 0].imshow(display_image)
        axes[row_index, 0].set_title(
            f"Index {index}: true G{true_grade}, predicted G{predicted_grade}"
        )
        axes[row_index, 1].imshow(display_image)
        axes[row_index, 1].imshow(native, cmap="jet", alpha=0.4)
        axes[row_index, 1].set_title("Native class map")
        axes[row_index, 2].imshow(display_image)
        axes[row_index, 2].imshow(gradient, cmap="jet", alpha=0.4)
        axes[row_index, 2].set_title("Final-feature Grad-CAM")
        for axis in axes[row_index]:
            axis.axis("off")
    figure.tight_layout()
    figure.savefig(
        SERESNEXT_BATCH_DIR / "native_vs_gradient_gradcam_gallery.png", dpi=160
    )
    plt.close(figure)

    method_table = [
        "| method | joint_enrichment | border_enrichment | peak_inside_joint |",
        "| --- | ---: | ---: | ---: |",
    ]
    for method, values in method_summary.iterrows():
        method_table.append(
            f"| {method} | {values['joint_enrichment']:.4f} | "
            f"{values['border_enrichment']:.4f} | "
            f"{values['peak_inside_joint']:.4f} |"
        )
    comparison_report = [
        "",
        "## Native CAM versus gradient Grad-CAM",
        "",
        "The native map is the production explanation because its spatial mean is the exact predicted CE logit. Gradient Grad-CAM is retained as a secondary diagnostic only.",
        "",
        *method_table,
        "",
        "Interpretation: map agreement and visual plausibility are not sufficient evidence of faithfulness; use the existing occlusion/logit-sensitivity audit to arbitrate disagreements.",
        "References: Zhou et al., CAM (https://arxiv.org/abs/1512.04150); Selvaraju et al., Grad-CAM (https://arxiv.org/abs/1610.02391); Chattopadhyay et al., Grad-CAM++ (https://arxiv.org/abs/1710.11063); Adebayo et al., saliency sanity checks (https://arxiv.org/abs/1810.03292); Li et al., guided attention supervision (https://arxiv.org/abs/1802.10171).",
    ]
    with open(
        SERESNEXT_BATCH_DIR / "seresnext_cam_ablation_report.md",
        "a",
        encoding="utf-8",
    ) as report_handle:
        report_handle.write("\n".join(comparison_report) + "\n")
    del selected_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("No promoted arm; gradient-CAM comparison skipped.")


Native versus Grad-CAM: 100%|██████████| 227/227 [00:23<00:00,  9.76it/s]


                  joint_enrichment  border_enrichment  peak_inside_joint
method                                                                  
gradient_gradcam          2.243711           0.238631                1.0
native_cam                2.243207           0.238941                1.0


## Release Colab Runtime


In [12]:
# Release Colab only after every checkpoint, CAM image, array, and report is saved.
try:
    from google.colab import runtime
    print("SE-ResNeXt CAM ablation complete. Releasing the Colab runtime...")
    runtime.unassign()
except ImportError:
    print("Not running in Google Colab; runtime release skipped.")


SE-ResNeXt CAM ablation complete. Releasing the Colab runtime...
